In [1]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoModel, AutoConfig, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Device: cpu


In [2]:
current_dir = os.getcwd()
split_dir = os.path.join(current_dir, "centralized_split")

df_train = pd.read_pickle(os.path.join(split_dir, "train.pkl"))
df_val   = pd.read_pickle(os.path.join(split_dir, "val.pkl"))

print(df_train.shape, df_val.shape)

feature_cols = ["input_ids", "attention_mask"]
label_cols = [c for c in df_train.columns if c not in feature_cols]

print("Label columns:", label_cols)
print("Num labels:", len(label_cols))


(9017, 14) (1932, 14)
Label columns: ['HS', 'Abusive', 'HS_Individual', 'HS_Group', 'HS_Religion', 'HS_Race', 'HS_Physical', 'HS_Gender', 'HS_Other', 'HS_Weak', 'HS_Moderate', 'HS_Strong']
Num labels: 12


In [3]:
class HateSpeechDataset(Dataset):
    def __init__(self, df, label_cols):
        self.input_ids = df["input_ids"].tolist()
        self.attention_mask = df["attention_mask"].tolist()
        self.labels = df[label_cols].values.astype("float32")

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.input_ids[idx], dtype=torch.long),
            "attention_mask": torch.tensor(self.attention_mask[idx], dtype=torch.long),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float),
        }

train_dataset = HateSpeechDataset(df_train, label_cols)
val_dataset   = HateSpeechDataset(df_val,   label_cols)

BATCH_SIZE = 8

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

len(train_loader), len(val_loader)


(1128, 242)

In [4]:
# Compute POSITIVE class weights (imbalanced dataset handling)
label_counts = df_train[label_cols].sum().values
pos_weight = (len(df_train) - label_counts) / (label_counts + 1e-6)
pos_weight = torch.tensor(pos_weight, dtype=torch.float).to(DEVICE)

print("pos_weight:", pos_weight)


pos_weight: tensor([ 1.3457,  1.5971,  2.6580,  5.5388, 15.2468, 22.1205, 38.8982, 41.5330,
         2.5099,  2.8584,  6.6221, 26.8302])


In [5]:
MODEL_NAME = "indobenchmark/indobert-base-p1"
NUM_LABELS = len(label_cols)

config = AutoConfig.from_pretrained(MODEL_NAME)
config.num_labels = NUM_LABELS
config.problem_type = "multi_label_classification"

base_model = AutoModel.from_pretrained(MODEL_NAME)

class IndoBertForMultiLabel(nn.Module):
    def __init__(self, base_model, num_labels):
        super().__init__()
        self.bert = base_model
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0]
        logits = self.classifier(self.dropout(cls))

        loss = None
        if labels is not None:
            loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
            loss = loss_fn(logits, labels)

        return loss, logits

model = IndoBertForMultiLabel(base_model, NUM_LABELS).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=2e-5)


In [6]:
EPOCHS = 2  # bisa naikkan ke 3 kalau kuat GPU

num_training_steps = len(train_loader) * EPOCHS
num_warmup_steps = int(0.1 * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

print("Training steps:", num_training_steps)


Training steps: 2256


In [7]:
def hamming_score(y_true, y_pred):
    acc_list = []
    for i in range(len(y_true)):
        set_true = set(np.where(y_true[i] == 1)[0])
        set_pred = set(np.where(y_pred[i] == 1)[0])
        acc_list.append(len(set_true & set_pred) / len(set_true | set_pred) if set_true | set_pred else 1)
    return sum(acc_list) / len(acc_list)


In [8]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0

    for batch in tqdm(loader, desc="Train"):
        optimizer.zero_grad()

        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        loss, logits = model(ids, mask, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def eval_one_epoch(model, loader, threshold=0.4):
    model.eval()
    total_loss = 0

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Validation"):
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            loss, logits = model(ids, mask, labels)
            total_loss += loss.item()

            probs = torch.sigmoid(logits)
            preds = (probs > threshold).int()

            all_labels.append(labels.cpu())
            all_preds.append(preds.cpu())

    y_true = torch.cat(all_labels).numpy()
    y_pred = torch.cat(all_preds).numpy()

    microF = f1_score(y_true, y_pred, average="micro")
    macroF = f1_score(y_true, y_pred, average="macro")
    hamming = hamming_score(y_true, y_pred)

    return total_loss / len(loader), microF, macroF, hamming


In [9]:
import numpy as np

def hamming_score(y_true, y_pred):
    acc_list = []
    for i in range(len(y_true)):
        true_set = set(np.where(y_true[i] == 1)[0])
        pred_set = set(np.where(y_pred[i] == 1)[0])

        if len(true_set) == 0 and len(pred_set) == 0:
            acc_list.append(1)  # both empty → perfect match
        else:
            acc_list.append(len(true_set & pred_set) / len(true_set | pred_set))

    return np.mean(acc_list)


In [10]:
for epoch in range(1, EPOCHS + 1):
    print(f"\n========== EPOCH {epoch}/{EPOCHS} ==========")

    train_loss = train_one_epoch(model, train_loader)
    val_loss, microF, macroF, hamming = eval_one_epoch(model, val_loader)

    print(f"Train loss : {train_loss:.4f}")
    print(f"Val loss   : {val_loss:.4f}")
    print(f"Micro F1   : {microF:.4f}")
    print(f"Macro F1   : {macroF:.4f}")
    print(f"Hamming    : {hamming:.4f}")



========== EPOCH 1/2 ==========


Validation: 100%|██████████| 242/242 [02:39<00:00,  1.51it/s]


Train loss : 0.7372
Val loss   : 0.4891
Micro F1   : 0.6352
Macro F1   : 0.5505
Hamming    : 0.5828

========== EPOCH 2/2 ==========


Validation: 100%|██████████| 242/242 [02:51<00:00,  1.41it/s]

Train loss : 0.3859
Val loss   : 0.4326
Micro F1   : 0.7108
Macro F1   : 0.6282
Hamming    : 0.6846


In [11]:
save_dir = os.path.join(current_dir, "models")
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "model_warmstart2.pt")
torch.save(model.state_dict(), save_path)

print("Saved:", save_path)


Saved: c:\Users\Alif\OneDrive - uinjkt.ac.id\Kuliah UIN\Skripsi\eksperimen skripsi\models\model_warmstart2.pt
